## MAGIC Gamma Telescope Classification

The obective of this analysis is to investigate whether logistic regression and SVMs can distinguish between gamma-ray and hadron events using measurements collected from the MAGIC Gamma Telescope dataset provided [here](https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope).

The target variable is `class`, which identifies whether the observed event is a gamma-ray event (`g`) or a hadron event (`h`) which makes the dataset suitable for binary classification.

## EDA

In [ ]:
# Imports of highest import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)
import time

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42

In [ ]:
columns = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
    "class"
]

df = pd.read_csv(
    "data/magic04.data",
    header=None,
    names=columns
)

print("Dataset dimensions:", df.shape)
print("\nNumber of observations:", df.shape[0])
print("Number of columns:", df.shape[1])
df.head()

In [ ]:
print(df["class"].value_counts())

print("\nClass proportions:")
print(df["class"].value_counts(normalize=True) * 100)
df.info()
df.isnull().sum()
df.describe().T

In [ ]:
features = df.drop(columns="class")
fig, axes = plt.subplots(5, 2, figsize=(15, 20))
for i, column in enumerate(features.columns):
    sns.histplot(
        data=df,
        x=column,
        kde=True,
        ax=axes[i // 2, i % 2]
    )
    axes[i // 2, i % 2].set_title(f"Distribution of {column}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(15, 22))
for i, column in enumerate(features.columns):
    sns.boxplot(
        data=df,
        x="class",
        y=column,
        hue="class",
        legend=False,
        ax=axes[i // 2, i % 2]
    )
    axes[i // 2, i % 2].set_title(f"{column} by class")
plt.tight_layout()
plt.show()


In [ ]:
correlation_matrix = features.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)
plt.title("Correlation Heatmap of Explanatory Variables")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(
    data=df,
    x="fLength",
    y="fWidth",
    hue="class",
    alpha=0.4,
    s=15,
    ax=axes[0]
)
axes[0].set_title("fLength vs fWidth")
sns.scatterplot(
    data=df,
    x="fAlpha",
    y="fDist",
    hue="class",
    alpha=0.4,
    s=15,
    ax=axes[1]
)
axes[1].set_title("fAlpha vs fDist")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=df,
    x="fAlpha",
    hue="class",
    fill=True,
    common_norm=False,
    alpha=0.3
)
plt.title("Distribution of fAlpha by Particle Class")
plt.xlabel("fAlpha")
plt.ylabel("Density")
plt.show()
df.groupby("class")[features.columns].median().T

### Observations

Chances are fAlpha will be our most promising indicator, and it should be noted that the strong correlation between some variables may indicate some overlapping information. The skewed distributions of fLength, fWidth, fSize, and fAlpha may also indicate some extreme observations to keep in mind. There's also a large class imbalance, so a classifier predicting gamma (`g`) for every observation would be 64.84% accurate. This is the minimum we need to beat!

## Logistic Regression

In [ ]:
X = df.drop(columns="class")
y = df["class"]

# 80/20 test split, seed is 42
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True))
print("\nTesting class proportions:")
print(y_test.value_counts(normalize=True))

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

# Keep track of training runtime
start_time = time.perf_counter()
logistic_model.fit(X_train, y_train)
logistic_training_time = time.perf_counter() - start_time

# Generate predictions
y_pred = logistic_model.predict(X_test)

# Calculate prediction probabilities
y_prob = logistic_model.predict_proba(X_test)[:, 1]

# Evaluate performance
print("Logistic Regression Results")
print("---------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label="g"))
print("Recall:", recall_score(y_test, y_pred, pos_label="g"))
print("F1-score:", f1_score(y_test, y_pred, pos_label="g"))
print("ROC-AUC:", roc_auc_score(y_test == "g", y_prob))
gamma_index = list(logistic_model.classes_).index("g")
y_prob = logistic_model.predict_proba(X_test)[:, gamma_index]
# Calculate ROC-AUC using gamma as the positive class
auc = roc_auc_score(
    (y_test == "g").astype(int),
    y_prob
)
print("Corrected ROC-AUC:", auc)

print("Training time:", logistic_training_time, "seconds")

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    logistic_model,
    X_test,
    y_test,
    cmap="Blues"
)

plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
# Extract logistic regression coefficients
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": logistic_model.named_steps["classifier"].coef_[0]
})

# Sort by absolute coefficient magnitude
coefficients["Absolute Coefficient"] = (
    coefficients["Coefficient"].abs()
)
coefficients = coefficients.sort_values(
    "Absolute Coefficient",
    ascending=False
)
print(coefficients)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=coefficients,
    x="Coefficient",
    y="Feature",
    hue="Coefficient",
    palette="coolwarm",
    legend=False
)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Logistic Regression Feature Coefficients")
plt.xlabel("Standardized Regression Coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, y_pred))
RocCurveDisplay.from_predictions(
    (y_test == "g").astype(int),
    y_prob
)

plt.title("Logistic Regression ROC Curve")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.show()

### Some observations

The initial results suggest that fLength and fAlpha are the most important predictors with fConc1, fM3Long, and fSize providing some additional information.

The model has high gamma recall but low hadron recall, which indicates that it's able to identify most gamma observations but misses quite a few hadron observations. We'll have to investigate class_weight='balanced' later.

## Linear SVM

In [ ]:
# Create Linear SVM pipeline
linear_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LinearSVC(
        C=1.0,
        max_iter=10000,
        random_state=RANDOM_STATE
    ))
])

# Train and measure runtime
start_time = time.perf_counter()

linear_svm.fit(X_train, y_train)
linear_svm_training_time = time.perf_counter() - start_time
svm_pred = linear_svm.predict(X_test)
svm_scores = linear_svm.decision_function(X_test)

# LinearSVC's positive decision scores correspond to classes_[1].
# Our desired positive class is gamma, classes_[0].
gamma_scores = -svm_scores

# Calculate metrics
print("Linear SVM Results")
print("------------------")
print("Accuracy:", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(
    y_test, svm_pred, pos_label="g"
))
print("Recall:", recall_score(
    y_test, svm_pred, pos_label="g"
))
print("F1-score:", f1_score(
    y_test, svm_pred, pos_label="g"
))
print("ROC-AUC:", roc_auc_score(
    (y_test == "g").astype(int),
    gamma_scores
))
print("Training time:", linear_svm_training_time, "seconds")

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    linear_svm,
    X_test,
    y_test,
    cmap="Blues"
)

plt.title("Linear SVM Confusion Matrix")
plt.show()

print(classification_report(y_test, svm_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

RocCurveDisplay.from_predictions(
    (y_test == "g").astype(int),
    y_prob,
    name="Logistic Regression",
    ax=ax
)

RocCurveDisplay.from_predictions(
    (y_test == "g").astype(int),
    gamma_scores,
    name="Linear SVM",
    ax=ax
)

ax.plot([0, 1], [0, 1], "--", color="grey")

plt.title("ROC Curve Comparison")
plt.show()

In [ ]:
results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label="g"),
        "Recall": recall_score(y_test, y_pred, pos_label="g"),
        "F1-score": f1_score(y_test, y_pred, pos_label="g"),
        "ROC-AUC": roc_auc_score(
            (y_test == "g").astype(int), y_prob
        ),
        "Training Time": logistic_training_time
    },
    {
        "Model": "Linear SVM",
        "Accuracy": accuracy_score(y_test, svm_pred),
        "Precision": precision_score(y_test, svm_pred, pos_label="g"),
        "Recall": recall_score(y_test, svm_pred, pos_label="g"),
        "F1-score": f1_score(y_test, svm_pred, pos_label="g"),
        "ROC-AUC": roc_auc_score(
            (y_test == "g").astype(int), gamma_scores
        ),
        "Training Time": linear_svm_training_time
    }
])

results.round(4)

## Class weight Investigation

As we've discovered, the classes are not well balanced at all, so lets check how changing class_weight to be balanced affects things. My assumption is that we'll see improved metrics for hadron observations at the expense of gamma observations.

In [ ]:
logistic_balanced = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

logistic_balanced.fit(X_train, y_train)

balanced_pred = logistic_balanced.predict(X_test)

print(classification_report(y_test, balanced_pred))

## Polynomial SVMs

We used a linear SVM earlier, but I wonder how overfitting these things with more complex polynomials would work out. I'm sure we'll see improved scores in the training data but the tests will be worse off.

In [ ]:
# Polynomial SVM model
poly_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="poly",
        degree=3,
        C=1.0,
        gamma="scale",
        class_weight=None
    ))
])

start_time = time.perf_counter()

poly_svm.fit(X_train, y_train)
poly_training_time = time.perf_counter() - start_time
poly_pred = poly_svm.predict(X_test)

# SVC decision scores are positive for hadron observations
poly_scores = -poly_svm.decision_function(X_test)

print("Polynomial SVM Results")
print("----------------------")
print("Accuracy:", accuracy_score(y_test, poly_pred))
print("Precision:", precision_score(
    y_test, poly_pred, pos_label="g"
))
print("Recall:", recall_score(
    y_test, poly_pred, pos_label="g"
))
print("F1-score:", f1_score(
    y_test, poly_pred, pos_label="g"
))
print("ROC-AUC:", roc_auc_score(
    (y_test == "g").astype(int),
    poly_scores
))
print("Training time:", poly_training_time)
print("----------------------")
print(classification_report(
    y_test,
    poly_pred
))

In [ ]:
# RBF SBF Model
rbf_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight=None
    ))
])

start_time = time.perf_counter()

rbf_svm.fit(X_train, y_train)
rbf_training_time = time.perf_counter() - start_time
rbf_pred = rbf_svm.predict(X_test)
rbf_scores = -rbf_svm.decision_function(X_test)

print("RBF SVM Results")
print("----------------")
print("Accuracy:", accuracy_score(y_test, rbf_pred))
print("Precision:", precision_score(
    y_test, rbf_pred, pos_label="g"
))
print("Recall:", recall_score(
    y_test, rbf_pred, pos_label="g"
))
print("F1-score:", f1_score(
    y_test, rbf_pred, pos_label="g"
))
print("ROC-AUC:", roc_auc_score(
    (y_test == "g").astype(int),
    rbf_scores
))
print("Training time:", rbf_training_time)
print("----------------------")
print(classification_report(
    y_test,
    rbf_pred
))

### Observations

It appears just from looking at the numbers that the models have improved their performance, but the runtime has gone up significantly compared to the linear SVM and logistic regression models. However, let's check the balanced class weights with the RBF SVMs to see if that improves what appears to be the best-performing model.

## RBF SVM Class Weight Check

In [ ]:
rbf_svm_balanced = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    ))
])

rbf_svm_balanced.fit(X_train, y_train)

rbf_balanced_pred = rbf_svm_balanced.predict(X_test)

print(classification_report(
    y_test,
    rbf_balanced_pred
))

### Observations

It appears just from looking at the numbers that the models improved their score, but the runtime has gone up significantly for both of these models and the improvement wasn't too much higher.

## Overall Results Comparison Table

In [ ]:
models = {
    "Logistic Regression": (logistic_model, y_pred),
    "Logistic Regression Balanced": (logistic_balanced, balanced_pred),
    "Linear SVM": (linear_svm, svm_pred),
    "Polynomial SVM": (poly_svm, poly_pred),
    "RBF SVM": (rbf_svm, rbf_pred),
    "RBF SVM Balanced": (rbf_svm_balanced, rbf_balanced_pred)
}

model_results = []

for name, (model, predictions) in models.items():

    scores = -model.decision_function(X_test)

    model_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, pos_label="g"
        ),
        "Recall": recall_score(
            y_test, predictions, pos_label="g"
        ),
        "F1-score": f1_score(
            y_test, predictions, pos_label="g"
        ),
        "Macro F1": f1_score(
            y_test, predictions, average="macro"
        ),
        "ROC-AUC": roc_auc_score(
            (y_test == "g").astype(int), scores
        )
    })

comparison = pd.DataFrame(model_results)

comparison.round(4)

In [ ]:
comparison_melted = comparison.melt(
    id_vars="Model",
    value_vars=[
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Macro F1",
        "ROC-AUC"
    ],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(13, 6))

sns.barplot(
    data=comparison_melted,
    x="Metric",
    y="Score",
    hue="Model"
)

plt.ylim(0, 1)
plt.title("Classification Performance Comparison")
plt.tight_layout()
plt.show()

## Cross Validation and Hyperparameters

At the moment, we're only evaluating with one split. Lets introduce cross validation and see how consistent the performance is. We shoudl also check if we're beholden to different hyperparameters.

In [ ]:
from sklearn.model_selection import (
    cross_validate,
    StratifiedKFold
)

# Define the cross-validation strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Models to evaluate
models = {
    "Logistic Regression": logistic_model,
    "Logistic Regression Balanced": logistic_balanced,
    "Linear SVM": linear_svm,
    "Polynomial SVM": poly_svm,
    "RBF SVM": rbf_svm,
    "RBF SVM Balanced": rbf_svm_balanced
}

# Metrics to evaluate
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc"
}

cv_results = []

for name, model in models.items():

    start_time = time.perf_counter()

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    training_time = time.perf_counter() - start_time

    result = {
        "Model": name,
        "Macro F1 Mean": scores["test_f1"].mean(),
        "Macro F1 Std": scores["test_f1"].std(),
        "Training Time": training_time,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),
        "Precision Mean": scores["test_precision"].mean(),
        "Recall Mean": scores["test_recall"].mean(),
        "ROC-AUC Mean": scores["test_roc_auc"].mean(),
        "ROC-AUC Std": scores["test_roc_auc"].std()
    }

    cv_results.append(result)

# Create comparison DataFrame
cv_comparison = pd.DataFrame(cv_results)

cv_comparison = cv_comparison.sort_values(
    by="Macro F1 Mean",
    ascending=False
)

cv_comparison.round(4)

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=cv_comparison,
    x="Model",
    y="Macro F1 Mean"
)

plt.errorbar(
    x=range(len(cv_comparison)),
    y=cv_comparison["Macro F1 Mean"],
    yerr=cv_comparison["Macro F1 Std"],
    fmt="none",
    color="black",
    capsize=5
)

plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.ylabel("Cross-Validation Macro F1")
plt.title("Model Comparison - 5-Fold Cross Validation")
plt.tight_layout()
plt.show()